# Channel 回归：AI Adoption 与 Attention

**目录**

| 节 | 内容 | 状态 |
|---|---|---|
| §1 | Baseline 模型：设定、变量定义、固定效应 | — |
| **§2** | **Channel 1：ADOPT（GenAI 采用）** | |
| §2.1 | 变量与方程 | |
| §2.2 | 不含 LLM signal 的基本回归 | 本轮 |
| §2.3 | 加入 LLM signal | 本轮 |
| **§3** | **Channel 2：ATT（注意力稀缺）** | |
| §3.1 | 变量与方程（NRANK / ATT 的构造） | |
| §3.2 | 不含 LLM signal 的基本回归 | 本轮 |
| §3.3 | 加入 LLM signal | 本轮 |
| §5 | LLM signal 单独进回归（不含 moderator） | 本轮 |
| §4 | LLM signal：数据来源、处理思路、产出与覆盖率 | 已建好 |

**上游依赖**

| 文件 | 由什么产生 | 内容 |
|---|---|---|
| `build/pead_panel.parquet` | `数据准备.ipynb`（8 步流水线） | 事件级大表，**75 列**，含 `n_ann_day` / `nrank` / `att`。构造细节见 `README.md` |
| `build/ctrl_att.parquet` | `数据准备.ipynb` **§6.2** | NRANK/ATT 的中间表，键为 `eid` |

NRANK / ATT 是事件的固有属性，已直接进大表。LLM signal 先做成独立 side table
（窗口与聚合方式还要反复试），窗口定下来后再固化进大表。

baseline 回归本身见 `回归准备.ipynb`；本 notebook 沿用完全相同的样本、控制变量与固定效应。

---

## 1. Baseline 模型

后面每个 channel 都是在这个方程上加项，所以先写清楚。

### 1.1 方程

$w \in \{ANN,\ DRIFT\}$，收益口径 $\in \{C2C,\ O2O\}$，每个设定 4 条回归：

$$CAR^{w}_{i,d}
=\alpha^{w}
+\underbrace{\beta^{w}_1\, SUE^{rank}_{i,d}}_{\text{核心}}
+\sum_{k=1}^{10}\gamma^{w}_k X_{k,i,d}
+\underbrace{\eta^{w}_t}_{\text{年+月+星期}}
+\underbrace{\psi^{w}_j}_{\text{FF10 行业}}
+\varepsilon^{w}_{i,d}$$

上标 $w \in \{ANN,\ DRIFT\}$ 标在每个系数上：两个窗口是**分开估计**的，
同一个 $\beta_1$ 在两个窗口下是两个不同的数。下文为省字，同一方程内只写一次上标说明。

- $CAR^{ANN}$：公告窗口 $[0,1]$ 的累计异常收益
- $CAR^{DRIFT}$：漂移窗口 $[2,61]$ 的累计异常收益
- 异常收益 = 个股 buy-and-hold 收益 − 同 size×B/M 25 组基准组合的 buy-and-hold 收益

### 1.2 变量定义

| 变量 | 全称 | 构造 | 取值 |
|---|---|---|---|
| `car_ann_*` | **CAR** = Cumulative Abnormal Return，**ANN** = Announcement window | $\prod_{k=d}^{d+1}(1+R_{i,k})-\prod_{k=d}^{d+1}(1+R_{p,k})$ | 小数（0.05 = 5%） |
| `car_drift_*` | **DRIFT** = post-announcement drift window | 同上，窗口换成 $[d+2,\ d+61]$ | 小数 |
| `sue` | **SUE** = Standardized Unexpected Earnings（HLT 2009 记作 **FE** = Forecast Error） | $(e-F)/P$：$e$ = 实际 EPS，$F$ = 公告前 60 个日历日内各分析师**最新**预测的**中位数**，$P$ = 拆股调整后的财季末股价 | 小数 |
| `sue_dec` | SUE decile | 按公告日所在**日历季度**独立分十份，1 = 最负意外，10 = 最正 | 1–10 |
| **`sue_rank`** | **进回归的核心自变量** | $(sue\_dec-1)/9$ | $[0,1]$ |
| `size_dec` | **SIZE** = Firm Size decile | formation 年 **6 月末**市值，NYSE 断点 | 1–10 |
| `bm_dec` | **BM** = Book-to-Market decile | $BE/ME$，BE = `SEQ+TXDITC−PS`，NYSE 断点 | 1–10 |
| `lnanalyst` | **LNANALYST** = Log(1 + # Analysts) | $\log(1+\#\{\text{公告前 365 天内出过预测的不同分析师}\})$ | ≥ 0 |
| `lag` `lag2` `lag3` | **LAG** = Reporting Lag | 公告日 − 财季结束日，及其平方、三次方 | 天 |
| `io` | **IO** = Institutional Ownership | 公告前最近一期 13F 的 $\sum\text{shares}/(\text{shrout}\times1000)$ | 0–1 |
| `evol` | **EVOL** = Earnings Volatility | 过去 16 财季 $\Delta_4 EPS$ 的样本标准差 | 美元/股 |
| `epersist` | **EPERSIST** = Earnings Persistence | 过去 16 财季**季度 EPS 水平值**的一阶自相关 | −1 ~ 1 |
| `turn` | **TURN** = Share Turnover | 过去 12 个月的月均 $\text{mthvol}/(\text{shrout}\times1000)$ | 小数 |

10 个控制变量的定义与 HLT 2009 §III.A 逐条一致。

### 1.3 固定效应与标准误

与 HLT 2009 Table III 相同：

| | 内容 | 吸收掉什么 |
|---|---|---|
| $\eta_t$ | 年 + 月 + 星期几 | 市场层面的时间效应、财报季节性、星期效应 |
| $\psi_j$ | FF10 行业（由 SIC 映射） | 行业层面的平均差异 |

$\beta_1$ 要回答的是"好消息公司 vs 坏消息公司"的横截面差异 —— PEAD 作为可交易异象，
交易者正是跨公司买卖，这个变异就是收益本身。公司层面的差异由 10 个控制变量
（含 SIZE、BM、IO、分析师覆盖）直接控住。

标准误按**公告日**聚类（CRV1），与 HLT 2009 相同。变量取原始值。

$\eta_t$ 含年固定效应，任何**纯时间断点**（如 ADOPT）的主效应会被它吸收，
只有与 SUE 的交互项可识别。§2.2 对此有专门处理。

In [1]:
import gc

import numpy as np
import pandas as pd
import pyfixest as pf

BUILD = "build"
panel = pd.read_parquet(f"{BUILD}/pead_panel.parquet")   # 含 n_ann_day / nrank / att

FF10 = [
    ("NoDur", [(100, 999), (2000, 2399), (2700, 2749), (2770, 2799), (3100, 3199), (3940, 3989)]),
    ("Durbl", [(2500, 2519), (2590, 2599), (3630, 3659), (3710, 3711), (3714, 3714), (3716, 3716),
               (3750, 3751), (3792, 3792), (3900, 3939), (3990, 3999)]),
    ("Manuf", [(2520, 2589), (2600, 2699), (2750, 2769), (3000, 3099), (3200, 3569), (3580, 3629),
               (3700, 3709), (3712, 3713), (3715, 3715), (3717, 3749), (3752, 3791), (3793, 3799),
               (3830, 3839), (3860, 3899)]),
    ("Enrgy", [(1200, 1399), (2900, 2999)]),
    ("HiTec", [(3570, 3579), (3660, 3692), (3694, 3699), (3810, 3829), (7370, 7379), (7391, 7391),
               (8730, 8734)]),
    ("Telcm", [(4800, 4899)]),
    ("Shops", [(5000, 5999), (7200, 7299), (7600, 7699)]),
    ("Hlth",  [(2830, 2839), (3693, 3693), (3840, 3859), (8000, 8099)]),
    ("Utils", [(4900, 4949)]),
]


def ff10(sic):
    s = pd.to_numeric(sic, errors="coerce")
    out = pd.Series("Other", index=s.index, dtype=object)
    done = pd.Series(False, index=s.index)
    for name, rngs in FF10:
        hit = pd.Series(False, index=s.index)
        for lo, hi in rngs:
            hit |= s.between(lo, hi)
        out[hit & ~done] = name
        done |= hit
    out[s.isna()] = np.nan
    return out


panel["ff10"] = ff10(panel["siccd"])

CTRL = ["size_dec", "bm_dec", "lnanalyst", "lag", "lag2", "lag3", "io", "evol", "epersist", "turn"]
CARS = {("ANN", "C2C"): "car_ann_c2c", ("DRIFT", "C2C"): "car_drift_c2c",
        ("ANN", "O2O"): "car_ann_o2o", ("DRIFT", "O2O"): "car_drift_o2o"}


# 与 回归准备.ipynb §B.1 完全相同的样本构造
df = panel[panel["is_latest_pends_on_day"] & ~panel["flag_lag_bad"]].copy()
df = df.dropna(subset=["sue", "sue_dec", "ff10"] + CTRL)
df = df.dropna(subset=["car_ann_c2c", "car_drift_c2c"])
df["sue_rank"] = (df["sue_dec"] - 1) / 9.0
df["date_id"] = pd.factorize(df["anndats"])[0]

print(f"Baseline 样本: {len(df):,} 个公告 | {df['permno'].nunique():,} 家公司 | "
      f"{df['anndats'].dt.year.min()}-{df['anndats'].dt.year.max()}")
del panel
gc.collect()

Baseline 样本: 302,952 个公告 | 11,152 家公司 | 1996-2026


10

In [2]:
from dataclasses import dataclass

FE_COLS = ["year", "month", "dow", "ff10"]          # follow HLT 2009 Table III
FE_FULL = " + ".join(FE_COLS)


@dataclass
class FitResult:
    tidy: pd.DataFrame
    n: int
    r2_within: float


def run(y, xs, fe=FE_FULL, data=None, cluster="date_id"):
    """跑一条固定效应回归。只把用到的列传给 pyfixest，避免复制整张大表。"""
    d = df if data is None else data
    fe_cols = [c.strip() for c in fe.split("+")]
    cl_cols = [c.strip() for c in cluster.split("+")]
    cols = list(dict.fromkeys([y] + list(xs) + fe_cols + cl_cols))
    slim = d[cols].dropna()
    fit = pf.feols(f"{y} ~ {' + '.join(xs)} | {fe}", data=slim, vcov={"CRV1": cluster})
    out = FitResult(fit.tidy(), int(fit._N), float(fit._r2_within))
    del fit, slim
    gc.collect()
    return out


def grab(res, var):
    t = res.tidy
    return (t.loc[var, "Estimate"], t.loc[var, "Std. Error"],
            t.loc[var, "t value"], t.loc[var, "Pr(>|t|)"])


def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""


def pub_table(fits, keep, labels=None, title="", notes="", fe_rows=None):
    """论文风格回归表：系数 + 星号，括号内标准误。"""
    labels = labels or {}
    cols, w = list(fits.keys()), 14
    line = "=" * (30 + w * len(cols))
    out = [title, line,
           f"{'':<30}" + "".join(f"{f'({i + 1})':>{w}}" for i in range(len(cols))),
           f"{'':<30}" + "".join(f"{f'{a} {b}':>{w}}" for a, b in cols),
           "-" * (30 + w * len(cols))]
    for v in keep:
        r1, r2 = f"{labels.get(v, v):<30}", f"{'':<30}"
        for c in cols:
            if v in fits[c].tidy.index:
                b, se, _, p = grab(fits[c], v)
                r1 += f"{f'{b:.4f}{stars(p)}':>{w}}"
                r2 += f"{f'({se:.4f})':>{w}}"
            else:
                r1 += f"{'':>{w}}"
                r2 += f"{'':>{w}}"
        out += [r1, r2]
    out.append("-" * (30 + w * len(cols)))
    for lbl in (fe_rows if fe_rows else ["Year / Month / DoW FE", "Industry FE (FF10)"]):
        out.append(f"{lbl:<30}" + "".join(f"{'Yes':>{w}}" for _ in cols))
    out.append(f"{'Controls':<30}" + "".join(f"{'Yes':>{w}}" for _ in cols))
    out.append(f"{'Observations':<30}" + "".join(f"{fits[c].n:>{w},}" for c in cols))
    out.append(f"{'Within R-squared':<30}" + "".join(f"{fits[c].r2_within:>{w}.4f}" for c in cols))
    out.append(line)
    if notes:
        out.append(notes)
    return "\n".join(out)


LABELS = {"__sep__": "Controls and interactions:", "sue_rank": "SUE rank [0,1]", "adopt": "ADOPT", "sue_x_adopt": "  SUE x ADOPT",
          "att": "ATT (11 - NRANK)", "sue_x_att": "  SUE x ATT",
          "nrank": "NRANK", "sue_x_nrank": "  FE x NRANK",
          "size_dec": "SIZE decile", "bm_dec": "BM decile", "lnanalyst": "LNANALYST",
          "lag": "LAG", "lag2": "LAG2", "lag3": "LAG3", "io": "IO", "evol": "EVOL",
          "epersist": "EPERSIST", "turn": "TURN"}
print(f"pyfixest {pf.__version__} | FE: {FE_FULL}")


# 控制变量与 SUE 的交互（HLT Table III 的做法）。控制变量先中心化：
# X_k 取值如 size_dec ∈ 1..10，不中心化的话 β1 变成"所有控制变量都等于 0 时"的效应，
# 而 0 不在样本内。中心化后 β1 = 控制变量取均值时 SUE 的效应，与不带交互的设定可比。
def add_ctrl_x(d, var, tag):
    """给 d 加上 CTRL 中每个变量与 var 的交互项，返回新列名。"""
    names = []
    for c in CTRL:
        nm = f"{c}_x_{tag}"
        d[nm] = (d[c].astype("float64") - d[c].astype("float64").mean()) * d[var]
        LABELS[nm] = f"  {LABELS[c]} x {tag.upper()}"
        names.append(nm)
    return names


INTER_SUE = add_ctrl_x(df, "sue_rank", "sue")
print(f"控制变量 × SUE 交互项 {len(INTER_SUE)} 个")

pyfixest 0.60.0 | FE: year + month + dow + ff10
控制变量 × SUE 交互项 10 个


---

# 2. Channel 1：ADOPT（GenAI 采用）

## 2.1 变量与方程

$$ADOPT_{d}=\mathbb{1}\{d \ge \text{2022-11-30}\}$$

2022-11-30 是 ChatGPT 公开发布日。这是一个**纯时间断点**：同一天的所有公司取值相同。

### 完整方程

$$
\begin{aligned}
CAR^{w}_{i,d}=\ &\alpha^{w}
+\beta^{w}_1 SUE^{rank}_{i,d}
+\beta^{w}_2 ADOPT_d
+\beta^{w}_3\left(SUE^{rank}_{i,d}\times ADOPT_d\right)\\
&+\beta^{w}_4 LLM_{i,d}
+\underbrace{\beta^{w}_5\left(ADOPT_d\times LLM_{i,d}\right)}_{\text{最终核心}}\\
&+\sum_k\gamma^{w}_k X_{k,i,d}
+\sum_k\delta^{w}_k\left(X_{k,i,d}\times SUE^{rank}_{i,d}\right)
+\sum_k\phi^{w}_k\left(X_{k,i,d}\times LLM_{i,d}\right)\\
&+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}_{i,d}
\end{aligned}
$$

**为什么控制变量必须交互**：这是 HLT 2009 eq. (4) 的形式，表注写明
"All control variables are interacted with FE"。这些变量本身就会改变市场对盈余的**敏感度**
（规模小、分析师少的公司反应更慢），不交互的话 $\beta^{w}_3$ 会把某个与 $ADOPT$ 相关的
控制变量的敏感度效应也算进来。要测 $\beta^{w}_5$ 时同理，控制变量还要与 $LLM$ 交互。

**本节（§2.2）先跑不含 $LLM$ 的部分**，即去掉 $\beta^{w}_4$、$\beta^{w}_5$、$\phi^{w}_k$ 三组：

$$CAR^{w}_{i,d}=\alpha^{w}+\beta^{w}_1 SUE^{rank}+\beta^{w}_2 ADOPT
+\underbrace{\beta^{w}_3\left(SUE^{rank}\times ADOPT\right)}_{\text{本节要看的}}
+\sum_k\gamma^{w}_k X_k+\sum_k\delta^{w}_k\left(X_k\times SUE^{rank}\right)
+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}$$

$\beta^{w}_3$ 回答：**ChatGPT 出现之后，盈余漂移变强还是变弱**。

### 系数的参照点

控制变量在交互前**先中心化**，所以 $\beta^{w}_1$ 是"控制变量取均值时"的 SUE 效应。
$ADOPT$ 是 0/1 哑变量，不中心化，故 $\beta^{w}_1$ 读作 **ChatGPT 之前**的 SUE 效应，
$\beta^{w}_1+\beta^{w}_3$ 是之后的。

### 年固定效应的处理

$\eta_t$ 含年固定效应，而 $ADOPT$ 是纯时间断点，两者高度共线 ——
$ADOPT$ 的主效应 $\beta^{w}_2$ 完全被年固定效应吸收（pyfixest 会直接把它从设计矩阵里剔除，
所以 Table A1 里没有这一行）。交互项 $\beta^{w}_3$ 不受影响：它靠的是**同一年内 SUE 高低组之间**的差异。

因此下面跑两个设定：

| | 固定效应 | 能读什么 |
|---|---|---|
| **(A)** | 年 + 月 + 星期 + FF10（baseline 全套） | 只读 $\beta^{w}_3$。这是主设定 |
| **(B)** | 月 + 星期 + FF10（**去掉年**） | $\beta^{w}_2$ 与 $\beta^{w}_3$ 都能读，代价是时间趋势不再被控住 |

## 2.2 不含 LLM signal 的基本回归

In [3]:
ADOPT_DATE = pd.Timestamp("2022-11-30")          # ChatGPT 公开发布
df["adopt"] = (df["anndats"] >= ADOPT_DATE).astype("float64")
df["sue_x_adopt"] = df["sue_rank"] * df["adopt"]

_n_post = int(df["adopt"].sum())
print(f"公告日 >= {ADOPT_DATE.date()} 的事件: {_n_post:,} ({_n_post / len(df):.1%})"
      f" | 涉及 {df.loc[df['adopt'] == 1, 'permno'].nunique():,} 家公司")
print(f"样本末端: {df['anndats'].max().date()}")
print("\n处理组 / 对照组的原始 CAR 均值（%）")
_cmp = (df.groupby("adopt")[list(CARS.values())].mean() * 100).round(3)
_cmp.index = ["pre  (before 2022-11-30)", "post (on/after 2022-11-30)"]
_cmp.columns = [f"{w} {c}" for w, c in CARS]
print(_cmp.to_string())

公告日 >= 2022-11-30 的事件: 33,072 (10.9%) | 涉及 3,301 家公司
样本末端: 2026-05-14

处理组 / 对照组的原始 CAR 均值（%）
                            ANN C2C  DRIFT C2C  ANN O2O  DRIFT O2O
pre  (before 2022-11-30)      0.063     -0.892    0.116     -1.380
post (on/after 2022-11-30)    0.095      0.016    0.197     -0.464


In [4]:
XS_A = ["sue_rank", "sue_x_adopt"] + CTRL + INTER_SUE          # 年 FE 吸收 adopt 主效应
XS_B = ["sue_rank", "adopt", "sue_x_adopt"] + CTRL + INTER_SUE
FE_NOYEAR = "month + dow + ff10"
KEEP_A = ["sue_rank", "adopt", "sue_x_adopt"]

res_a1 = {k: run(y, XS_A) for k, y in CARS.items()}
KEEP_FULL_A = ["sue_rank", "sue_x_adopt", "__sep__"] + CTRL + INTER_SUE
print(pub_table(res_a1, ["sue_rank", "sue_x_adopt"], LABELS,
                title="Table A1. ADOPT channel, specification (A): full baseline fixed effects",
                notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                      "ADOPT = 1 if the announcement falls on or after 2022-11-30 (ChatGPT release).\n"
                      "Its main effect is absorbed by the year fixed effects; only the interaction is identified.\n"
                      "All ten controls enter directly and interacted with SUE, following HLT (2009) Table III.\n"
                      "Controls are demeaned before interacting, so the SUE coefficient is the effect at mean\n"
                      "covariates and in the pre-ChatGPT period. Dependent variable: CAR in decimals; SUE rank is the\n"
                      "within-quarter decile rescaled to [0,1]."))

res_a2 = {k: run(y, XS_B, fe=FE_NOYEAR) for k, y in CARS.items()}
print("\n" + pub_table(res_a2, ["sue_rank", "adopt", "sue_x_adopt"], LABELS,
                       title="Table A2. ADOPT channel, specification (B): year fixed effects removed",
                       notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                             "Dropping year fixed effects makes the ADOPT main effect estimable, at the cost of\n"
                             "leaving secular time trends uncontrolled. Compare the interaction with Table A1.",
                       fe_rows=["Month / DoW FE", "Industry FE (FF10)"]))

# 不带控制变量交互的版本，看 β3 是否稳
res_a0 = {k: run(y, ["sue_rank", "sue_x_adopt"] + CTRL) for k, y in CARS.items()}
print("\nSUE x ADOPT across specifications")
_r = [{"Specification": lbl,
       **{f"{w} {c}": f"{grab(rr[(w, c)], 'sue_x_adopt')[0]:.4f}"
                      f"{stars(grab(rr[(w, c)], 'sue_x_adopt')[3])}" for w, c in CARS}}
      for lbl, rr in [("(A) year FE, controls x SUE", res_a1),
                      ("(B) no year FE, controls x SUE", res_a2),
                      ("(C) year FE, controls not interacted", res_a0)]]
print(pd.DataFrame(_r).to_string(index=False))

Table A1. ADOPT channel, specification (A): full baseline fixed effects
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0803***     0.0237***     0.0751***     0.0325***
                                    (0.0007)      (0.0018)      (0.0006)      (0.0018)
  SUE x ADOPT                      0.0096***       -0.0006     0.0092***       -0.0026
                                    (0.0026)      (0.0071)      (0.0022)      (0.0081)
--------------------------------------------------------------------------------------
Year / Month / DoW FE                    Yes           Yes           Yes           Yes
Industry FE (FF10)                       Yes           Yes           Yes           Yes
Controls                                 Yes           Yes


Table A2. ADOPT channel, specification (B): year fixed effects removed
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0802***     0.0240***     0.0750***     0.0327***
                                    (0.0007)      (0.0018)      (0.0006)      (0.0018)
ADOPT                             -0.0046***        0.0063    -0.0032***        0.0059
                                    (0.0014)      (0.0040)      (0.0012)      (0.0047)
  SUE x ADOPT                      0.0097***       -0.0013     0.0094***       -0.0034
                                    (0.0026)      (0.0072)      (0.0022)      (0.0083)
--------------------------------------------------------------------------------------
Month / DoW FE                           Yes           Yes


SUE x ADOPT across specifications
                       Specification   ANN C2C DRIFT C2C   ANN O2O DRIFT O2O
         (A) year FE, controls x SUE 0.0096***   -0.0006 0.0092***   -0.0026
      (B) no year FE, controls x SUE 0.0097***   -0.0013 0.0094***   -0.0034
(C) year FE, controls not interacted 0.0116***   -0.0086 0.0108***   -0.0104


**全系数表**（上面的表只保留关键系数，便于阅读）

In [5]:
# 全系数版：上面的表只列了关键系数，这里把控制变量与全部交互项一并列出
print(pub_table(res_a1, ["sue_rank", "sue_x_adopt", "__sep__"] + CTRL + INTER_SUE, LABELS,
                title="Table A1-full. ADOPT channel, specification (A): all coefficients",
                notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                      "Controls are demeaned before interacting with SUE."))

print("\n" + pub_table(res_a2, ["sue_rank", "adopt", "sue_x_adopt", "__sep__"] + CTRL + INTER_SUE, LABELS,
                       title="Table A2-full. ADOPT channel, specification (B): all coefficients",
                       notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                             "Controls are demeaned before interacting with SUE.",
                       fe_rows=["Month / DoW FE", "Industry FE (FF10)"]))

Table A1-full. ADOPT channel, specification (A): all coefficients
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0803***     0.0237***     0.0751***     0.0325***
                                    (0.0007)      (0.0018)      (0.0006)      (0.0018)
  SUE x ADOPT                      0.0096***       -0.0006     0.0092***       -0.0026
                                    (0.0026)      (0.0071)      (0.0022)      (0.0081)
Controls and interactions:                                                            
                                                                                      
SIZE decile                        0.0024***     0.0019***     0.0019***     0.0038***
                                    (0.0002)      (0.0005)      

In [6]:
# 直接估 pre / post 两段的 SUE 斜率，而不是只看交互项 ——
# 交互项不显著只说明"两段的差"分不出来，分段斜率才能看出漂移在 post 期是否还在
df["sue_pre"] = df["sue_rank"] * (1 - df["adopt"])
df["sue_post"] = df["sue_rank"] * df["adopt"]
seg = {k: run(y, ["sue_pre", "sue_post"] + CTRL + INTER_SUE) for k, y in CARS.items()}
LABELS.update({"sue_pre": "SUE rank, pre-ChatGPT", "sue_post": "SUE rank, post-ChatGPT"})
print(pub_table(seg, ["sue_pre", "sue_post"], LABELS,
                title="Table A3. SUE slope estimated separately for the two periods",
                notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                      "Same specification as Table A1, with the SUE slope split at 2022-11-30 instead of\n"
                      "entering an interaction term. Reading the two slopes directly shows whether the drift\n"
                      "is still present after the break, which the interaction alone cannot tell."))
print("\n" + pub_table(seg, ["sue_pre", "sue_post", "__sep__"] + CTRL + INTER_SUE, LABELS,
                       title="Table A3-full. Period-specific SUE slopes: all coefficients",
                       notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                             "Controls are demeaned before interacting with SUE."))
df.drop(columns=["sue_pre", "sue_post"], inplace=True)

Table A3. SUE slope estimated separately for the two periods
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank, pre-ChatGPT              0.0803***     0.0237***     0.0751***     0.0325***
                                    (0.0007)      (0.0018)      (0.0006)      (0.0018)
SUE rank, post-ChatGPT             0.0899***     0.0232***     0.0843***     0.0299***
                                    (0.0025)      (0.0068)      (0.0021)      (0.0078)
--------------------------------------------------------------------------------------
Year / Month / DoW FE                    Yes           Yes           Yes           Yes
Industry FE (FF10)                       Yes           Yes           Yes           Yes
Controls                                 Yes           Yes           

### 2.2 的完整解读

**结果**

| | ANN C2C | DRIFT C2C | ANN O2O | DRIFT O2O |
|---|---|---|---|---|
| $\beta^{w}_3$（SUE × ADOPT） | **+0.0096\*\*\*** | −0.0006 | **+0.0092\*\*\*** | −0.0026 |
| 标准误 | (0.0026) | (0.0071) | (0.0022) | (0.0081) |

**① 公告窗口的反应显著变强。** ChatGPT 之后 D10−D1 的即时价差多出 0.96 个百分点（C2C）、
0.92 个百分点（O2O）。Table A3 分段直接估斜率看得更清楚：
C2C 从 **0.0803 升到 0.0899**（+12%），O2O 从 0.0751 升到 0.0843（+12%）。
方向与"AI 工具帮助投资者更快消化盈余信息"一致。

**② 漂移窗口：符号朝预期方向，但查不出变化。**
$\beta^{DRIFT}_3 = -0.0006$（C2C）、$-0.0026$（O2O），符号为负 —— 与"更快打进价格 → 漂移变小"
一致，但都不显著。标准误 0.0071 / 0.0081，95% 置信区间宽到把"漂移减半"和"漂移翻倍"
都包含在内。post 期只有 33,072 个事件（占 10.9%），漂移窗口还要 61 个交易日，检出力不足。

**③ 漂移在 post 期依然存在。** Table A3：post 期的漂移斜率是 **0.0232\*\*\***（C2C）、
**0.0299\*\*\***（O2O），都显著为正，与 pre 期的 0.0237 / 0.0325 基本持平。
所以正确的说法是"**没有证据显示漂移变小**"，而不是"漂移消失了"。

**④ 整体图景。** 公告当下的反应变强（+12%，显著）、漂移方向上略降但仍显著存在 ——
这半边符合"AI 之后信息捕捉更到位"，另半边（漂移应随之缩小）还看不到。

**⑤ 识别上的限制。** ADOPT 是纯日历断点，2022 年底同期发生的其他变化
（加息周期、熊市结束、样本构成）都与它共线，单靠这个断点无法把功劳归给 GenAI。
这正是 Design Doc 把核心检验放在 $\beta^{w}_5 = ADOPT \times LLM$ 的原因 ——
LLM signal 提供**公司层面的横截面变异**，可以在同一时点比较"被 AI 关注多"与
"被 AI 关注少"的公司，把纯时间效应差分掉。

## 2.3 加入 LLM signal

### 变量

$$LLM^{[a,b]}_{i,d}=\prod_{t=d+a}^{d+b}\bigl(1+\overline{pred}_{i,t}\bigr)-1$$

$\overline{pred}_{i,t}$ 是公司 $i$ 在交易日 $t$ 所有新闻 `pred` 的平均。
窗口内各日的 $(1+\overline{pred})$ **连乘再减 1**，与 $CAR$ 的 buy-and-hold 同法，
实现上用 $\log(1+\overline{pred})$ 在交易日轴上累积再 $\exp(\cdot)-1$（构造细节见 §4）。

两个窗口，都以公告的第一个交易日 $d$ 为锚：

| 后缀 | 窗口 | 交易日数 | 说明 |
|---|---|---|---|
| `0_1` | $[d,\ d+1]$ | **2 天** | 与 $CAR^{ANN}$ 完全同窗口 |
| `m1_1` | $[d-1,\ d+1]$ | **3 天** | 多含公告前一个交易日，捕捉盘前与提前泄露 |

### 标准化

`build/llm_signal.parquet` 里存的是**原始的** `llm_cum_*`（收益量纲，标准差约 0.002）。
标准化在本 notebook 的回归准备 cell 里做，不写回文件 —— 这样以后换口径
（例如改用 `llm_rank_*`）不必重建文件。

对每个窗口，用**该窗口全样本**的均值与标准差：

```python
x = df[f"llm_cum_{tag}"].astype("float64")
df[f"llm_{tag}"] = (x - x.mean()) / x.std()
```

| 窗口 | 有信号事件 | 均值 | 标准差 |
|---|---|---|---|
| $[d,\ d+1]$ | 140,608 | +0.000808 | 0.001842 |
| $[d-1,\ d+1]$ | 141,325 | +0.001005 | 0.002087 |

所以 $\beta^{w}_4$、$\beta^{w}_5$ 读作"**LLM signal 每高 1 个标准差**"。
这是单调线性变换：交互项系数对中心化不变，$\beta^{w}_5$ 的取值与显著性不受这一步影响。

### 完整方程

$$
\begin{aligned}
CAR^{w}_{i,d}=\ &\alpha^{w}
+\beta^{w}_1 SUE^{rank}_{i,d}
+\beta^{w}_2 ADOPT_d
+\beta^{w}_3\left(SUE^{rank}_{i,d}\times ADOPT_d\right)\\
&+\beta^{w}_4 LLM_{i,d}
+\beta^{w}_6\left(SUE^{rank}_{i,d}\times LLM_{i,d}\right)
+\underbrace{\beta^{w}_5\left(ADOPT_d\times LLM_{i,d}\right)}_{\text{核心}}\\
&+\sum_k\gamma^{w}_k X_{k,i,d}
+\sum_k\delta^{w}_k\left(X_{k,i,d}\times SUE^{rank}_{i,d}\right)
+\sum_k\phi^{w}_k\left(X_{k,i,d}\times LLM_{i,d}\right)\\
&+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}_{i,d}
\end{aligned}
$$

$\beta^{w}_5$ 问的是：**ChatGPT 之后，LLM signal 对 CAR 的作用是否变了**。
$ADOPT$ 是纯时间断点，主效应 $\beta^{w}_2$ 被年固定效应吸收，与 §2.2 相同。

$\{SUE,\ M,\ LLM\}$ 三个变量的**两两交互全部放进方程**：$SUE\times M$、$SUE\times LLM$、$M\times LLM$。
三者相关，只放其中两个会让被遗漏的那个污染 $\beta^{w}_5$ 的估计 ——
$SUE\times LLM$ 在 §5 里对漂移有 1% 显著的正效应，不能省。

样本限制在窗口内有新闻的事件（`llm_cum` 非缺失）。

In [7]:
llm = pd.read_parquet(f"{BUILD}/llm_signal.parquet")
df = df.merge(llm[[c for c in llm.columns if c.startswith("llm_")] + ["eid"]], on="eid", how="left")

LLM_WINDOWS = {"0_1": "[d, d+1]", "m1_1": "[d-1, d+1]"}

# pred 的训练目标是 future_1d_O2O_RET（open-to-open），O2O 口径的 CAR 与它同源。
# 主表四个口径都报，另外把 O2O 单独列一次。
CARS_O2O = {k: v for k, v in CARS.items() if k[1] == "O2O"}

# llm_cum 的标准差只有 ~0.0018，标准化成"1 个标准差"为单位，系数才好读。
# 这是单调线性变换：交互项系数的 t 值与显著性完全不受影响。
for tag in LLM_WINDOWS:
    x = df[f"llm_cum_{tag}"].astype("float64")
    df[f"llm_{tag}"] = (x - x.mean()) / x.std()
    LABELS[f"llm_{tag}"] = "LLM signal (1 s.d.)"
    print(f"llm_cum_{tag} {LLM_WINDOWS[tag]}: 有信号 {int(x.notna().sum()):,} 个事件 | "
          f"均值 {x.mean():+.6f} 标准差 {x.std():.6f}")


def llm_terms(d, tag, m_var):
    """LLM 主效应 + 三个变量的两两交互（SUE×LLM、M×LLM）+ 10 个 X_k × LLM。

    {SUE, M, LLM} 三者相关，方程里已有 SUE×M，若漏掉 SUE×LLM，
    被遗漏的交互会污染 M×LLM 的估计，所以两个都放。
    """
    z = d[f"llm_{tag}"]
    d[f"{m_var}_x_llm"] = d[m_var] * z
    d["sue_x_llm"] = d["sue_rank"] * z
    LABELS[f"{m_var}_x_llm"] = f"  {m_var.upper()} x LLM"
    LABELS["sue_x_llm"] = "  SUE x LLM"
    names = [f"llm_{tag}", "sue_x_llm", f"{m_var}_x_llm"]
    for c in CTRL:
        nm = f"{c}_x_llm"
        d[nm] = (d[c].astype("float64") - d[c].astype("float64").mean()) * z
        LABELS[nm] = f"  {LABELS[c]} x LLM"
        names.append(nm)
    return names

llm_cum_0_1 [d, d+1]: 有信号 140,608 个事件 | 均值 +0.000808 标准差 0.001842
llm_cum_m1_1 [d-1, d+1]: 有信号 141,325 个事件 | 均值 +0.001005 标准差 0.002087


In [8]:
res_llm_a, res_llm_a_full = {}, {}
for tag, lab in LLM_WINDOWS.items():
    d_ = df[df[f"llm_cum_{tag}"].notna()].copy()
    LL = llm_terms(d_, tag, "adopt")
    xs = ["sue_rank", "sue_x_adopt"] + CTRL + INTER_SUE + LL
    keep = ["sue_rank", "sue_x_adopt", f"llm_{tag}", "sue_x_llm", "adopt_x_llm"]
    r = {k: run(y, xs, data=d_) for k, y in CARS.items()}
    res_llm_a[tag] = r
    res_llm_a_full[tag] = (xs, keep)
    print(pub_table(r, keep, LABELS,
                    title=f"Table A4[{tag}]. ADOPT x LLM signal, window {lab} ({abs(int(tag.split('_')[1]) - (-int(tag.split('_')[0][1:]) if tag.split('_')[0].startswith('m') else int(tag.split('_')[0]))) + 1} trading days)",
                    notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                          "LLM signal = compounded prediction over the window, standardised to mean 0 and s.d. 1.\n"
                          "ADOPT's main effect is absorbed by the year fixed effects.\n"
                          "All ten controls enter directly, interacted with SUE, and interacted with LLM.\n"
                          "pred is trained on the open-to-open return, so the O2O columns are the matched convention;\n"
                          "C2C is reported alongside for comparison.\n"
                          "Sample: events with at least one news item in the window."))
    print()
    del d_
    gc.collect()

def window_summary(res, var, cars, title):
    rows = []
    for tag, lab in LLM_WINDOWS.items():
        k0 = list(cars)[0]
        row = {"Window": f"{lab} ({tag})", "N": f"{res[tag][k0].n:,}"}
        for w, c in cars:
            b, se, t, p = grab(res[tag][(w, c)], var)
            row[f"{w} {c}"] = f"{b:+.4f}{stars(p)} ({t:+.1f})"
        rows.append(row)
    print(title)
    print(pd.DataFrame(rows).to_string(index=False))


window_summary(res_llm_a, "adopt_x_llm", CARS, "ADOPT x LLM across the two windows")
print("\nO2O only (the convention pred is trained on)")
window_summary(res_llm_a, "adopt_x_llm", CARS_O2O, "")

Table A4[0_1]. ADOPT x LLM signal, window [d, d+1] (2 trading days)
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0758***     0.0132***     0.0686***     0.0219***
                                    (0.0012)      (0.0033)      (0.0014)      (0.0034)
  SUE x ADOPT                      0.0112***        0.0085     0.0107***        0.0059
                                    (0.0031)      (0.0089)      (0.0025)      (0.0092)
LLM signal (1 s.d.)                0.0160***       -0.0021     0.0165***       -0.0018
                                    (0.0007)      (0.0016)      (0.0006)      (0.0016)
  SUE x LLM                        0.0045***      0.0068**     0.0028***     0.0087***
                                    (0.0011)      (0.0027)    

Table A4[m1_1]. ADOPT x LLM signal, window [d-1, d+1] (3 trading days)
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0782***     0.0159***     0.0714***     0.0238***
                                    (0.0011)      (0.0037)      (0.0012)      (0.0036)
  SUE x ADOPT                      0.0143***        0.0107     0.0138***        0.0086
                                    (0.0030)      (0.0087)      (0.0025)      (0.0091)
LLM signal (1 s.d.)                0.0148***      -0.0030*     0.0152***      -0.0028*
                                    (0.0007)      (0.0016)      (0.0006)      (0.0016)
  SUE x LLM                        0.0031***     0.0092***      0.0021**     0.0107***
                                    (0.0011)      (0.0027) 

**全系数表**

In [9]:
# 全系数版
for tag, lab in LLM_WINDOWS.items():
    xs, _ = res_llm_a_full[tag]
    print(pub_table(res_llm_a[tag],
                    ["sue_rank", "sue_x_adopt", f"llm_{tag}", "sue_x_llm", "adopt_x_llm", "__sep__"] + xs[2:],
                    LABELS,
                    title=f"Table A4[{tag}]-full. All coefficients, window {lab}",
                    notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01"))
    print()

Table A4[0_1]-full. All coefficients, window [d, d+1]
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0758***     0.0132***     0.0686***     0.0219***
                                    (0.0012)      (0.0033)      (0.0014)      (0.0034)
  SUE x ADOPT                      0.0112***        0.0085     0.0107***        0.0059
                                    (0.0031)      (0.0089)      (0.0025)      (0.0092)
LLM signal (1 s.d.)                0.0160***       -0.0021     0.0165***       -0.0018
                                    (0.0007)      (0.0016)      (0.0006)      (0.0016)
  SUE x LLM                        0.0045***      0.0068**     0.0028***     0.0087***
                                    (0.0011)      (0.0027)      (0.0009)    

### 2.3 的解读

**核心系数 $\beta^{w}_5$（ADOPT × LLM，每 1 个标准差）**

| 窗口 | N | ANN C2C | DRIFT C2C | ANN O2O | DRIFT O2O |
|---|---|---|---|---|---|
| $[d,\ d+1]$ | 140,608 | +0.0013\* (1.6) | +0.0009 (0.4) | **+0.0015\*\*** (2.1) | +0.0011 (0.5) |
| $[d-1,\ d+1]$ | 141,325 | **−0.0023\*\*\*** (−2.8) | −0.0001 (−0.1) | **−0.0016\*\*** (−2.2) | −0.0006 (−0.3) |

**① 两个窗口在 ANN 上符号相反，这个结果不成立。**
$[d,d+1]$ 是 +0.0015\*\*，$[d-1,d+1]$ 是 −0.0016\*\*，两个都过了 5%，方向却相反。
两个窗口只差公告前一个交易日，`llm_cum` 相关系数高达 0.92 ——
说明这个交互项由**公告前那一天**的信号主导，且与公告当天方向相反。
在弄清这一天为什么起相反作用之前，$\beta^{ANN}_5$ 不能作为结论。

**② DRIFT 上四个格子全不显著**，$|t|\le 0.5$。这是 Design Doc 真正要检验的那一格，
目前没有证据。post 期只有三年多，再叠加"窗口内有新闻"这个筛选，检出力不足。

**③ 加入 $SUE\times LLM$ 后结论未变。** 补上这个被遗漏的交互前，
$[d,d+1]$ 的 ANN O2O 是 +0.0014\*、$[d-1,d+1]$ 是 −0.0016\*\*；补上后是 +0.0015\*\* 与 −0.0016\*\*。
符号相反这件事不是设定不全造成的。

**④ 结合 §2.2 看。** 不含 LLM 时 ADOPT 就已经识别不出来（纯日历断点、与同期其他变化共线），
加入 LLM 后样本还砍掉一半（302,952 → 约 14 万）。这个 channel 需要的是更好的
采用度量（连续的采用强度而非 0/1 断点），而不是更多的交互项。

---

# 3. Channel 2：ATT（注意力稀缺）

## 3.1 变量与方程

HLT 2009 用**同日公告数**度量投资者被分散的注意力。由 **`数据准备.ipynb` §6.2** 构造，
中间表 `build/ctrl_att.parquet`，三列已并入大表（`n_ann_day` / `nrank` / `att`），两步：

1. **每日公告总数** `n_ann_day`：来自**全部** Compustat 季报的 `rdq`（1,118,065 条，
   不限 I/B/E/S 覆盖，与原文一致）；若 I/B/E/S 的公告日更早则取更早的那个
   （原文规则，影响 2.7% 的记录）
2. **NRANK**：在每个**日历季度**内，把该季度的事件按其公告当日的公告总数排十分位，
   1 = 当日公告最少，10 = 当日公告最多（= 最分心）

$$ATT_{i,d} = 11 - NRANK_{i,d} \in \{1,\dots,10\}$$

**ATT 越大 = 注意力越充裕**（同日竞争公告越少）。与原文的 NRANK 方向相反，方便读符号。
实测每日公告数中位数 72，论文 Table I Panel A 报 71。

### 完整方程

$$
\begin{aligned}
CAR^{w}_{i,d}=\ &\alpha^{w}
+\beta^{w}_1 SUE^{rank}_{i,d}
+\beta^{w}_2 ATT_{i,d}
+\beta^{w}_3\left(SUE^{rank}_{i,d}\times ATT_{i,d}\right)\\
&+\beta^{w}_4 LLM_{i,d}
+\underbrace{\beta^{w}_5\left(ATT_{i,d}\times LLM_{i,d}\right)}_{\text{最终核心}}\\
&+\sum_k\gamma^{w}_k X_{k,i,d}
+\sum_k\delta^{w}_k\left(X_{k,i,d}\times SUE^{rank}_{i,d}\right)
+\sum_k\phi^{w}_k\left(X_{k,i,d}\times LLM_{i,d}\right)\\
&+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}_{i,d}
\end{aligned}
$$

控制变量与盈余意外的交互是 HLT 2009 eq. (4) 的形式：这些变量本身会改变市场对盈余的
**敏感度**，不交互的话 $\beta^{w}_3$ 会把它们的敏感度效应算进来。

**本节（§3.2）先跑不含 $LLM$ 的部分**：

$$CAR^{w}_{i,d}=\alpha^{w}+\beta^{w}_1 SUE^{rank}+\beta^{w}_2 ATT
+\underbrace{\beta^{w}_3\left(SUE^{rank}\times ATT\right)}_{\text{本节要看的}}
+\sum_k\gamma^{w}_k X_k+\sum_k\delta^{w}_k\left(X_k\times SUE^{rank}\right)
+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}$$

控制变量交互前先中心化，$ATT$ 保持 1–10 的原始刻度（与原文的 NRANK 同刻度，便于对照），
所以 $\beta^{w}_1$ 的参照点是 $ATT=0$ —— 落在取值范围之外，只作代数意义上的截距，
真正要读的是 $\beta^{w}_3$。

### 预期符号

原文的假设用 NRANK 表述（NRANK 越大越分心）：公告窗口的反应**更弱**（$a_3<0$）、
漂移**更强**（$a_3>0$）。我们的 ATT = 11 − NRANK，符号全部翻转：

| 窗口 | 原文（NRANK） | 我们（ATT） | 含义 |
|---|---|---|---|
| ANN $[0,1]$ | $a_3 < 0$ | $\beta^{ANN}_3 > 0$ | 注意力充裕 → 公告当下反应更强 |
| DRIFT $[2,61]$ | $a_3 > 0$ | $\beta^{DRIFT}_3 < 0$ | 注意力充裕 → 后续漂移更弱 |

原文 Table III 第 (2)(4) 列的 $a_3$ 是 −0.015 与 +0.049（FE 为 1–10 整数刻度）。

## 3.2 不含 LLM signal 的基本回归

In [10]:
d_att = df[df["att"].notna()].copy()
d_att["sue_x_att"] = d_att["sue_rank"] * d_att["att"]

print(f"ATT 可得的样本: {len(d_att):,} / {len(df):,} ({len(d_att) / len(df):.1%})")
print("\n各 NRANK 十分位的当日公告数与 CAR（%）")
_t = d_att.groupby("nrank").agg(n=("eid", "size"), ann_per_day=("n_ann_day", "median"))
_t = _t.join((d_att.groupby("nrank")[list(CARS.values())].mean() * 100).round(3))
_t.columns = ["N", "announcements that day (median)"] + [f"{w} {c}" for w, c in CARS]
_t.index = [f"NRANK {int(i)} (ATT {11 - int(i)})" for i in _t.index]
print(_t.to_string())

ATT 可得的样本: 302,685 / 302,952 (99.9%)

各 NRANK 十分位的当日公告数与 CAR（%）
                      N  announcements that day (median)  ANN C2C  DRIFT C2C  ANN O2O  DRIFT O2O
NRANK 1 (ATT 10)  26072                             43.0    0.057     -0.943    0.147     -1.451
NRANK 2 (ATT 9)   29130                             96.0    0.274     -0.829    0.287     -1.207
NRANK 3 (ATT 8)   30334                            159.0    0.081     -0.894    0.222     -1.448
NRANK 4 (ATT 7)   30812                            224.0    0.125     -0.913    0.162     -1.370
NRANK 5 (ATT 6)   31792                            287.0    0.093     -0.994    0.111     -1.478
NRANK 6 (ATT 5)   32550                            338.0    0.090     -0.524    0.196     -1.070
NRANK 7 (ATT 4)   31874                            383.0    0.038     -0.673    0.127     -1.252
NRANK 8 (ATT 3)   31852                            431.0   -0.051     -0.676    0.012     -1.181
NRANK 9 (ATT 2)   31689                            478.0   -0.0

In [11]:
INTER_SUE_ATT = add_ctrl_x(d_att, "sue_rank", "sue")
XS_ATT = ["sue_rank", "att", "sue_x_att"] + CTRL + INTER_SUE_ATT
KEEP_ATT = ["sue_rank", "att", "sue_x_att"]

res_att = {k: run(y, XS_ATT, data=d_att) for k, y in CARS.items()}
print(pub_table(res_att, KEEP_ATT, LABELS,
                title="Table B1. Attention channel: ATT = 11 - NRANK",
                notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                      "ATT is 11 minus the decile rank of the number of same-day earnings announcements,\n"
                      "so a higher value means more investor attention available. Predicted signs on the\n"
                      "interaction: positive for ANN, negative for DRIFT.\n"
                      "All ten controls enter directly and interacted with SUE, following HLT (2009) Table III.\n"
                      "Dependent variable: CAR in decimals. SUE rank is the within-quarter decile rescaled to [0,1],\n"
                      "so its coefficient reads as the D10-minus-D1 difference."))

res_att0 = {k: run(y, ["sue_rank", "att", "sue_x_att"] + CTRL, data=d_att) for k, y in CARS.items()}
_chk = []
for (w, c) in CARS:
    b, se, t, p = grab(res_att[(w, c)], "sue_x_att")
    b0 = grab(res_att0[(w, c)], "sue_x_att")[0]
    want = "positive" if w == "ANN" else "negative"
    _chk.append({"Window": w, "Returns": c, "SUE x ATT": f"{b:.4f}{stars(p)}", "t": round(t, 1),
                 "without controls x SUE": f"{b0:.4f}",
                 "Predicted": want, "Match": "yes" if (b > 0) == (want == "positive") else "no"})
print("\nSign check against the distraction hypothesis")
print(pd.DataFrame(_chk).to_string(index=False))

Table B1. Attention channel: ATT = 11 - NRANK
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0711***     0.0281***     0.0645***     0.0386***
                                    (0.0015)      (0.0043)      (0.0013)      (0.0044)
ATT (11 - NRANK)                  -0.0008***       -0.0004    -0.0009***       -0.0002
                                    (0.0002)      (0.0005)      (0.0001)      (0.0005)
  SUE x ATT                        0.0019***       -0.0009     0.0022***      -0.0012*
                                    (0.0003)      (0.0007)      (0.0002)      (0.0007)
--------------------------------------------------------------------------------------
Year / Month / DoW FE                    Yes           Yes           Yes           Y


Sign check against the distraction hypothesis
Window Returns SUE x ATT    t without controls x SUE Predicted Match
   ANN     C2C 0.0019***  7.5                 0.0015  positive   yes
 DRIFT     C2C   -0.0009 -1.2                -0.0010  negative   yes
   ANN     O2O 0.0022*** 10.0                 0.0017  positive   yes
 DRIFT     O2O  -0.0012* -1.7                -0.0013  negative   yes


In [12]:
# 论文刻度对照：NRANK 原始方向、FE 用 1-10 整数、CAR 用百分点、控制变量全部与 FE 交互
d_att["sue_dec_int"] = d_att["sue_dec"].astype("float64")
d_att["sue_x_nrank"] = d_att["sue_dec_int"] * d_att["nrank"]
for c in CTRL:
    d_att[f"{c}_xFE"] = d_att[c].astype("float64") * d_att["sue_dec_int"]
for col in CARS.values():
    d_att[f"{col}_pct"] = d_att[col] * 100

XS_P = ["sue_dec_int", "nrank", "sue_x_nrank"] + CTRL + [f"{c}_xFE" for c in CTRL]
LAB_P = dict(LABELS, sue_dec_int="FE (earnings surprise decile 1-10)")

res_p = {k: run(f"{col}_pct", XS_P, fe="year + month + dow + ff10",
                data=d_att, cluster="date_id") for k, col in CARS.items()}
print(pub_table(res_p, ["sue_dec_int", "nrank", "sue_x_nrank"], LAB_P,
                title="Table B2. HLT (2009) Table III specification, replicated on 1996-2026",
                notes="Dependent variable: CAR in percentage points. FE is the earnings surprise decile (1-10),\n"
                      "NRANK the number-of-announcements decile. Standard errors clustered by announcement date.\n"
                      "No firm fixed effects and all controls interacted with FE, following the paper.\n"
                      "Published estimates (1995-2004, N = 112,839): FE x NRANK = -0.015 for CAR[0,1]\n"
                      "and +0.049 for CAR[2,61].",
                fe_rows=["Year / Month / DoW FE", "Industry FE (FF10)"]))

Table B2. HLT (2009) Table III specification, replicated on 1996-2026
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
FE (earnings surprise decile 1-10)     0.8566***     0.7989***     0.7185***     1.0056***
                                    (0.0669)      (0.1870)      (0.0559)      (0.1879)
NRANK                              0.1039***        0.0266     0.1143***        0.0030
                                    (0.0176)      (0.0526)      (0.0151)      (0.0532)
  FE x NRANK                      -0.0212***        0.0094    -0.0242***       0.0135*
                                    (0.0028)      (0.0076)      (0.0024)      (0.0078)
--------------------------------------------------------------------------------------
Year / Month / DoW FE                    Yes           Y

**全系数表**

In [13]:
# 全系数版
print(pub_table(res_att, KEEP_ATT + ["__sep__"] + CTRL + INTER_SUE_ATT, LABELS,
                title="Table B1-full. Attention channel: all coefficients",
                notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                      "Controls are demeaned before interacting with SUE."))

print("\n" + pub_table(res_p, ["sue_dec_int", "nrank", "sue_x_nrank", "__sep__"] + CTRL
                        + [f"{c}_xFE" for c in CTRL], LAB_P,
                        title="Table B2-full. HLT (2009) specification: all coefficients",
                        notes="Standard errors clustered by announcement date in parentheses.\n"
                              "CAR in percentage points; FE is the earnings surprise decile (1-10).",
                        fe_rows=["Year / Month / DoW FE", "Industry FE (FF10)"]))

Table B1-full. Attention channel: all coefficients
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0711***     0.0281***     0.0645***     0.0386***
                                    (0.0015)      (0.0043)      (0.0013)      (0.0044)
ATT (11 - NRANK)                  -0.0008***       -0.0004    -0.0009***       -0.0002
                                    (0.0002)      (0.0005)      (0.0001)      (0.0005)
  SUE x ATT                        0.0019***       -0.0009     0.0022***      -0.0012*
                                    (0.0003)      (0.0007)      (0.0002)      (0.0007)
Controls and interactions:                                                            
                                                                               

### 3.2 的完整解读

**① 公告窗口完全符合 distraction 假说，漂移窗口方向一致。**

| | ANN C2C | DRIFT C2C | ANN O2O | DRIFT O2O |
|---|---|---|---|---|
| $\beta^{w}_3$（SUE × ATT） | **+0.0019\*\*\*** | −0.0009 | **+0.0022\*\*\*** | **−0.0012\*** |
| 预期 | 正 | 负 | 正 | 负 |

注意力充裕（同日竞争公告少）时，公告当下对盈余意外的反应更强 —— 两个口径都在 1% 水平显著。
漂移端符号都为负，O2O 口径边际显著（t = −1.7），C2C 不显著。

**② 与 HLT 2009 的定量对照。** §3.3 的 Table B2 与原文刻度、设定完全相同，可以直接比：

| | 本项目 1996–2026 | HLT 2009 原文 1995–2004 |
|---|---|---|
| $FE \times NRANK$，CAR[0,1] | **−0.0205\*\*\*** (0.0023) | **−0.015** (0.005) |
| $FE \times NRANK$，CAR[2,61] | +0.0089 (0.0060) | **+0.049** (0.019) |
| 观测数 | 302,685 | 112,839 |

公告窗口完全复制，量级甚至略强。漂移窗口方向一致但量级约为原文的五分之一。

**③ 漂移端衰减的原因是样本期。** PEAD 本身在 2000 年后大幅减弱
（`回归准备.ipynb` §D 的 "Post-2000 only" 一行）。distraction 效应是作用在**漂移**上的
调节效应 —— 被调节的量本身缩小，交互项自然一起缩小。原文样本 1995–2004 正好覆盖
PEAD 最强的那段。

**④ 结论：这个 channel 复制成功，可以直接往上接 LLM signal。**
ATT 覆盖 1996–2026，prediction 覆盖 2004 至 2026-03，交集充分。

## 3.3 加入 LLM signal

### 变量

$LLM$ 的构造与 §2.3 完全相同：窗口内各日 $(1+\overline{pred})$ 连乘再减 1，
标准化为均值 0、标准差 1。两个窗口 $[d,\ d+1]$（2 个交易日）与 $[d-1,\ d+1]$（3 个交易日）。

### 完整方程

$$
\begin{aligned}
CAR^{w}_{i,d}=\ &\alpha^{w}
+\beta^{w}_1 SUE^{rank}_{i,d}
+\beta^{w}_2 ATT_{i,d}
+\beta^{w}_3\left(SUE^{rank}_{i,d}\times ATT_{i,d}\right)\\
&+\beta^{w}_4 LLM_{i,d}
+\beta^{w}_6\left(SUE^{rank}_{i,d}\times LLM_{i,d}\right)
+\underbrace{\beta^{w}_5\left(ATT_{i,d}\times LLM_{i,d}\right)}_{\text{核心}}\\
&+\sum_k\gamma^{w}_k X_{k,i,d}
+\sum_k\delta^{w}_k\left(X_{k,i,d}\times SUE^{rank}_{i,d}\right)
+\sum_k\phi^{w}_k\left(X_{k,i,d}\times LLM_{i,d}\right)\\
&+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}_{i,d}
\end{aligned}
$$

$\beta^{w}_5$ 问的是：**注意力充裕时，LLM signal 对 CAR 的作用是否更强**。
如果 AI 可读的信息只有在投资者有余力处理时才被打进价格，
$\beta^{ANN}_5$ 应当为正、$\beta^{DRIFT}_5$ 为负。

与 §2.3 一样，$\{SUE,\ ATT,\ LLM\}$ 的两两交互全部放进方程。

样本限制在窗口内有新闻的事件。

In [14]:
res_llm_b, res_llm_b_full = {}, {}
for tag, lab in LLM_WINDOWS.items():
    d_ = df[df["att"].notna() & df[f"llm_cum_{tag}"].notna()].copy()
    d_["sue_x_att"] = d_["sue_rank"] * d_["att"]
    INTER_ATT = add_ctrl_x(d_, "sue_rank", "sue")
    LL = llm_terms(d_, tag, "att")
    xs = ["sue_rank", "att", "sue_x_att"] + CTRL + INTER_ATT + LL
    keep = ["sue_rank", "att", "sue_x_att", f"llm_{tag}", "sue_x_llm", "att_x_llm"]
    r = {k: run(y, xs, data=d_) for k, y in CARS.items()}
    res_llm_b[tag] = r
    res_llm_b_full[tag] = (xs, keep)
    print(pub_table(r, keep, LABELS,
                    title=f"Table B3[{tag}]. ATT x LLM signal, window {lab}",
                    notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                          "LLM signal = compounded prediction over the window, standardised to mean 0 and s.d. 1.\n"
                          "ATT = 11 - NRANK, so a higher value means more investor attention available.\n"
                          "All ten controls enter directly, interacted with SUE, and interacted with LLM.\n"
                          "pred is trained on the open-to-open return, so the O2O columns are the matched convention;\n"
                          "C2C is reported alongside for comparison.\n"
                          "Sample: events with at least one news item in the window."))
    print()
    del d_
    gc.collect()

window_summary(res_llm_b, "att_x_llm", CARS, "ATT x LLM across the two windows")
print("\nO2O only (the convention pred is trained on)")
window_summary(res_llm_b, "att_x_llm", CARS_O2O, "")

Table B3[0_1]. ATT x LLM signal, window [d, d+1]
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0658***        0.0082     0.0577***     0.0207***
                                    (0.0024)      (0.0072)      (0.0021)      (0.0068)
ATT (11 - NRANK)                  -0.0009***       -0.0009    -0.0011***       -0.0003
                                    (0.0002)      (0.0008)      (0.0002)      (0.0007)
  SUE x ATT                        0.0022***        0.0014     0.0025***        0.0006
                                    (0.0004)      (0.0012)      (0.0003)      (0.0011)
LLM signal (1 s.d.)                0.0176***        0.0030     0.0180***        0.0027
                                    (0.0010)      (0.0027)      (0.0008)      (0.

Table B3[m1_1]. ATT x LLM signal, window [d-1, d+1]
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0690***        0.0095     0.0608***     0.0225***
                                    (0.0024)      (0.0070)      (0.0021)      (0.0067)
ATT (11 - NRANK)                  -0.0010***       -0.0008    -0.0012***       -0.0002
                                    (0.0002)      (0.0008)      (0.0002)      (0.0007)
  SUE x ATT                        0.0022***        0.0012     0.0025***        0.0004
                                    (0.0004)      (0.0012)      (0.0003)      (0.0011)
LLM signal (1 s.d.)                0.0143***        0.0008     0.0149***        0.0003
                                    (0.0011)      (0.0028)      (0.0009)      

**全系数表**

In [15]:
# 全系数版
for tag, lab in LLM_WINDOWS.items():
    xs, _ = res_llm_b_full[tag]
    print(pub_table(res_llm_b[tag],
                    ["sue_rank", "att", "sue_x_att", f"llm_{tag}", "sue_x_llm", "att_x_llm", "__sep__"] + xs[3:],
                    LABELS,
                    title=f"Table B3[{tag}]-full. All coefficients, window {lab}",
                    notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01"))
    print()

Table B3[0_1]-full. All coefficients, window [d, d+1]
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0658***        0.0082     0.0577***     0.0207***
                                    (0.0024)      (0.0072)      (0.0021)      (0.0068)
ATT (11 - NRANK)                  -0.0009***       -0.0009    -0.0011***       -0.0003
                                    (0.0002)      (0.0008)      (0.0002)      (0.0007)
  SUE x ATT                        0.0022***        0.0014     0.0025***        0.0006
                                    (0.0004)      (0.0012)      (0.0003)      (0.0011)
LLM signal (1 s.d.)                0.0176***        0.0030     0.0180***        0.0027
                                    (0.0010)      (0.0027)      (0.0008)    

### 3.3 的解读

**核心系数 $\beta^{w}_5$（ATT × LLM，每 1 个标准差）**

| 窗口 | N | ANN C2C | DRIFT C2C | ANN O2O | DRIFT O2O |
|---|---|---|---|---|---|
| $[d,\ d+1]$ | 140,535 | −0.0001 (−0.8) | **−0.0007\*\*** (−2.1) | −0.0001 (−0.8) | **−0.0006\*** (−1.9) |
| $[d-1,\ d+1]$ | 141,246 | +0.0001 (0.5) | −0.0006 (−1.6) | +0.0001 (0.6) | −0.0005 (−1.4) |

**① DRIFT 上四个格子符号一致为负，与 distraction 假说的方向相符。**
$ATT$ 越大（同日竞争公告越少、注意力越充裕），LLM signal 留给后续漂移的部分越少。
逻辑链条是：新闻里的信息要被打进价格需要投资者有余力处理，
注意力充裕时当期就消化掉，注意力被分散时拖到公告之后。

**② 强度证据弱：只有 $[d,d+1]$ 窗口达到常规显著性**（C2C 5%、O2O 10%），
$[d-1,d+1]$ 两个口径的 $t$ 都在 1.5 附近。

**③ 加入 $SUE\times LLM$ 之后比原来更弱。**
补这个被遗漏的交互之前，$[d,d+1]$ 的 DRIFT 是 −0.0007\*\*（C2C，$t=-2.1$）
与 −0.0007\*\*（O2O，$t=-2.0$）；补上之后 O2O 降到 −0.0006\*（$t=-1.9$），
$[d-1,d+1]$ 的 C2C 从 10% 显著掉到不显著。
说明原先那部分显著性有一部分来自被遗漏的 $SUE\times LLM$。

**④ 量级参考。** 以 $[d,d+1]$、C2C 为例，$ATT$ 从最低档（1）升到最高档（10），
LLM signal 每标准差带来的漂移减少 $9\times0.0007=0.63$ 个百分点。
方向与量级都合理，但统计强度不足以称为证实。

**⑤ ANN 上接近 0（$|t|\le 0.8$）。**
按 distraction 的逻辑，注意力充裕时公告当期反应应当更强，$\beta^{ANN}_5$ 该为正。
数据没有支持这一边。一个可能的原因见 §5.3：$CAR^{ANN}$（O2O）与 `pred` 的训练目标区间重合，
$\beta^{ANN}_4$ 已经高达 +0.018，这部分机械关系不随注意力变化，会把交互项往 0 压。

**⑥ 与 §3.2 的对照。** 不含 LLM 时 `SUE × ATT` 在 ANN 显著为正、DRIFT 为负；
加入 LLM 后 `SUE × ATT` 的符号与显著性基本不变，说明 LLM signal 与 SUE 携带的是**不同的**信息，
两条渠道并存而非互相替代。

---

# 4. LLM signal

## 4.1 数据来源

**只读**，本项目不写入该目录：

```
/project/dachxiu/yifei/news/experiment/US/ARTICLE/RidgeProximal/
QUESTION_CHOICE_pred_1d_is4cv3_cossim_0.8_O2O_RET_future_1d_O2O_RET
_not_rank_normed_rolling_move_trading_days_expectation_only/
    pred_2004.pkl … pred_2026.pkl        逐年一个 DataFrame
```

**新闻级**数据，索引 `(timestamp, PERMNO)`，2004-01-02 至 2026-03-30。

| 列 | 含义 |
|---|---|
| `timestamp` | 新闻时间（ET，精确到毫秒） |
| `PERMNO` | 公司 |
| **`DATE`** | 新闻对应的**可交易日**。盘后新闻已被推到次日（16:00 那批占比最大），实测 100% 落在 CRSP 交易日历内，我们直接采用 |
| **`pred`** | **LLM signal**：新闻 embedding 经 Ridge 预测的**该交易日 open → 次日 open** 收益 |
| `labels` = `future_1d_O2O_RET` | 同一天的**实现**收益（firm-day 级） |
| `RIC` `takeSequence` `messageType` `urgency` | 单条新闻的元数据 |

**一天可以有多条新闻**：约三成 firm-day 有多行（2008 / 2015 / 2023 分别是 31.3% / 32.9% / 29.6%）。
同一 firm-day 内 `pred` 100% 各不相同、`labels` 100% 只有一个值 —— 这两列粒度不同，
正说明 `pred` 是逐条新闻的、`labels` 是那天的实现值。极端例子：波音 2019-03-13（737 MAX 停飞）
一天 48 条新闻。

## 4.2 处理思路

`pred` 的定义决定了每一步怎么聚合：**它是一个单日收益的预测值**。

**① 同一天多条新闻 → 取平均。**
同一天的几条新闻预测的是**同一个**单日收益：

$$\overline{pred}_{i,t}=\operatorname{mean}\{pred:\ \text{PERMNO}=i,\ \text{DATE}=t\}$$

**② 窗口内跨天 → 连乘。**
窗口覆盖多个交易日，每天的 $\overline{pred}$ 预测的是**不同的**单日收益。
收益的累计是连乘，与 $CAR$ 的 buy-and-hold 同法：

$$LLM^{[a,b]}_{i,d}=\prod_{t=d+a}^{d+b}\bigl(1+\overline{pred}_{i,t}\bigr)-1$$

实现上用 $\log(1+\overline{pred})$ 累积再 $\exp(\cdot)-1$，与 `build_car_path.py` 算 CAR 的写法一致。

**③ 另存每日均值与天数。**
`llm_mean` = 各天 $\overline{pred}$ 的算术平均，`llm_ndays` = 窗口内有新闻的交易日数。
`llm_cum` 把"信号强度"和"被报道的天数"合在一起，`llm_mean` 把天数除掉，
`llm_ndays` 让这个维度可以在回归里单独控住。

**④ 缺失与 0 分开。**
某天没新闻 = LLM 对那天没有预测，贡献 0；整个窗口都没新闻 = 这个事件**没有** LLM signal，
是缺失。所以 `n = 0` 时信号列记 `NaN`，`n` 与 `ndays` 记 0。

`pred` 保留符号：正 = LLM 看涨，负 = 看空，方向本身就是信号。

## 4.3 产出

由 **`build_llm_signal.py`** 生成，两层：

**第一层 `build/llm_daily.parquet`** —— 公司 × 交易日，2,296,215 行，8,436 家公司，
2004-01-02 ~ 2026-03-30

| 列 | 含义 |
|---|---|
| `permno` | 公司 |
| `date` | 可交易日 |
| `td_idx` | 该日在**全局交易日历**上的下标（0–7672），与大表 `td0_idx` 同一套日历 |
| `n_news` | 当天该公司的新闻条数 |
| `pred_mean` | 当天所有 `pred` 的平均 = **这一天的 LLM signal** |

**第二层 `build/llm_signal.parquet`** —— 一行一个事件，517,955 行 × 13 列，键为 `eid`

窗口后缀直接编码偏移量：`0_1` = $[d,\ d+1]$，`m1_1` = $[d-1,\ d+1]$（`m` = minus）。

| 列 | 含义 | 无新闻时 |
|---|---|---|
| `llm_cum_0_1` | 窗口内各天 $(1+\overline{pred})$ **连乘再减 1** = LLM 对 $[d,d+1]$ 累计收益的预测，与 $CAR[0,1]$ 同量纲 | `NaN` |
| `llm_mean_0_1` | 各天 `pred_mean` 的**算术平均**（只对有新闻的天） | `NaN` |
| `llm_ndays_0_1` | 窗口内**有新闻的交易日数** | 0 |
| `llm_n_0_1` | 窗口内新闻**总条数** | 0 |
| `llm_rank_0_1` | `llm_cum_0_1` 按公告所在**日历季度**排十分位再映到 $[0,1]$，与 `sue_rank` 同构 | `NaN` |
| `llm_*_m1_1` | 同样五列，窗口换成 $[d-1,\ d+1]$ | 同 |

**加窗口只要改一行**（`build_llm_signal.py` 顶部），第一层不用重建，第二层秒级重跑：

```python
WINDOWS = [(0, 1), (-1, 1)]      # 加 (-5, -1)、(2, 61) 即可
```

窗口运算是按 permno 在交易日轴上对 $\log(1+\overline{pred})$ 做累积和、取 `cum[t+b] − cum[t+a−1]` 再 `expm1`，任意窗口长度都是 O(1)。

## 4.4 覆盖率

语料从 2004 年起，所以覆盖率只在 **2004 年及以后**的样本上算。下面的 cell 逐年列出。

覆盖本身与公司规模、分析师关注度相关，所以 `llm_n > 0` 这个覆盖指示变量该进回归 ——
既是控制，也可能是一个 moderator。

## 4.5 接入后的完整方程

$M \in \{ADOPT,\ ATT\}$：

$$
\begin{aligned}
CAR^{w}_{i,d}=\ &\alpha^{w}
+\beta^{w}_1 SUE^{rank}+\beta^{w}_2 M+\beta^{w}_3(SUE^{rank}\times M)
+\beta^{w}_4 LLM+\underbrace{\beta^{w}_5(M\times LLM)}_{\text{核心}}\\
&+\sum_k\gamma^{w}_k X_k+\sum_k\delta^{w}_k(X_k\times SUE^{rank})+\sum_k\phi^{w}_k(X_k\times LLM)
+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}
\end{aligned}
$$

比 §2.2 / §3.2 多出三组：$LLM$ 主效应、$M\times LLM$（核心）、10 个 $X_k\times LLM$（系数 $\phi^{w}_k$）。

### 两个产出文件长什么样

In [16]:
llm_daily = pd.read_parquet(f"{BUILD}/llm_daily.parquet")

print(f"llm_daily.parquet: {len(llm_daily):,} 行 × {llm_daily.shape[1]} 列 | "
      f"{llm_daily['permno'].nunique():,} 家公司 | "
      f"{llm_daily['date'].min().date()} ~ {llm_daily['date'].max().date()}\n")
print(llm_daily.dtypes.to_frame("dtype").to_string())
print("\n前 5 行")
print(llm_daily.head().to_string(index=False))
print("\n每个 firm-day 的新闻条数分布")
print(llm_daily["n_news"].describe(percentiles=[.5, .75, .9, .99]).round(2).to_string())
print(f"  只有 1 条的占 {(llm_daily['n_news'] == 1).mean():.1%}")
print("\npred_mean（当天的 LLM signal）")
print(llm_daily["pred_mean"].describe(percentiles=[.01, .25, .5, .75, .99]).round(6).to_string())

llm_daily.parquet: 2,296,215 行 × 5 列 | 8,436 家公司 | 2004-01-02 ~ 2026-03-30

                    dtype
permno              int32
date       datetime64[ns]
td_idx              int32
n_news              int16
pred_mean         float64

前 5 行
 permno       date  td_idx  n_news  pred_mean
  10001 2013-06-18    4395       1   0.000170
  10001 2013-07-10    4410       1   0.000375
  10001 2013-07-22    4418       1   0.000518
  10001 2013-08-05    4428       1   0.000186
  10001 2013-09-26    4465       1  -0.000216

每个 firm-day 的新闻条数分布
count    2296215.00
mean           1.53
std            1.09
min            1.00
50%            1.00
75%            2.00
90%            3.00
99%            6.00
max           81.00
  只有 1 条的占 67.9%

pred_mean（当天的 LLM signal）


count    2.296215e+06
mean     9.560000e-04
std      1.233000e-03
min     -9.135000e-03
1%      -2.584000e-03
25%      3.640000e-04
50%      9.730000e-04
75%      1.626000e-03
99%      4.115000e-03
max      1.081600e-02


In [17]:
llm = pd.read_parquet(f"{BUILD}/llm_signal.parquet")

print(f"llm_signal.parquet: {len(llm):,} 行 × {llm.shape[1]} 列，键为 eid\n")
print(llm.dtypes.to_frame("dtype").to_string())

print("\n有信号的事件各列示例（前 5 行）")
print(llm[llm["llm_n_0_1"] > 0].head().to_string(index=False))
print("\n无信号的事件（信号列为 NaN，计数列为 0）")
print(llm[llm["llm_n_0_1"] == 0].head(3).to_string(index=False))

print("\n各列分布")
print(llm.drop(columns=["eid", "permno", "td0_idx"]).describe(percentiles=[.25, .5, .75]).T
        .round(6).to_string())

print("\n两个窗口的对比")
for tag, lab in [("0_1", "[d, d+1]"), ("m1_1", "[d-1, d+1]")]:
    m = llm[f"llm_n_{tag}"] > 0
    print(f"  {lab:<12} 有信号 {int(m.sum()):>7,} 个事件 | "
          f"平均 {llm.loc[m, f'llm_n_{tag}'].mean():.2f} 条新闻 / "
          f"{llm.loc[m, f'llm_ndays_{tag}'].mean():.2f} 个交易日 | "
          f"llm_cum 中位数 {llm.loc[m, f'llm_cum_{tag}'].median():+.6f}")
print(f"\n两个窗口的 llm_cum 相关系数: {llm['llm_cum_0_1'].corr(llm['llm_cum_m1_1']):.3f}")

llm_signal.parquet: 517,955 行 × 13 列，键为 eid

                  dtype
eid               int64
permno            int32
td0_idx           int32
llm_n_0_1         int32
llm_ndays_0_1      int8
llm_cum_0_1     float64
llm_mean_0_1    float64
llm_n_m1_1        int32
llm_ndays_m1_1     int8
llm_cum_m1_1    float64
llm_mean_m1_1   float64
llm_rank_0_1    float64
llm_rank_m1_1   float64

有信号的事件各列示例（前 5 行）
   eid  permno  td0_idx  llm_n_0_1  llm_ndays_0_1  llm_cum_0_1  llm_mean_0_1  llm_n_m1_1  llm_ndays_m1_1  llm_cum_m1_1  llm_mean_m1_1  llm_rank_0_1  llm_rank_m1_1
179357   74500     2017          1              1    -0.000052     -0.000052           1               1     -0.000052      -0.000052      0.000000       0.000000
179362   34948     2018          2              1     0.001385      0.001385           4               2      0.002123       0.001061      0.888889       1.000000
179369   75976     2018          1              1     0.001053      0.001053           2               2      0

                   count      mean       std       min       25%       50%       75%         max
llm_n_0_1       517955.0  1.105465  1.890625  0.000000  0.000000  0.000000  2.000000   76.000000
llm_ndays_0_1   517955.0  0.459978  0.660775  0.000000  0.000000  0.000000  1.000000    2.000000
llm_cum_0_1     189502.0  0.000736  0.001830 -0.012145 -0.000230  0.000793  0.001724    0.012645
llm_mean_0_1    189502.0  0.000606  0.001417 -0.007084 -0.000200  0.000685  0.001435    0.008166
llm_n_m1_1      517955.0  1.219691  2.109648  0.000000  0.000000  0.000000  2.000000  130.000000
llm_ndays_m1_1  517955.0  0.547887  0.812812  0.000000  0.000000  0.000000  1.000000    3.000000
llm_cum_m1_1    191068.0  0.000910  0.002055 -0.012341 -0.000148  0.000942  0.002003    0.015703
llm_mean_m1_1   191068.0  0.000657  0.001356 -0.007084 -0.000112  0.000723  0.001438    0.008166
llm_rank_0_1    189502.0  0.499946  0.319252  0.000000  0.222222  0.444444  0.777778    1.000000
llm_rank_m1_1   191068.0  0.49

  [d, d+1]     有信号 189,502 个事件 | 平均 3.02 条新闻 / 1.26 个交易日 | llm_cum 中位数 +0.000793
  [d-1, d+1]   有信号 191,068 个事件 | 平均 3.31 条新闻 / 1.49 个交易日 | llm_cum 中位数 +0.000942

两个窗口的 llm_cum 相关系数: 0.920


### 逐年覆盖率

In [18]:
# 覆盖率：只在 2004 年及以后的 baseline 回归样本上算
_cov = df[["eid", "anndats"]].merge(llm[["eid", "llm_n_0_1", "llm_n_m1_1"]], on="eid", how="left")
_cov = _cov[_cov["anndats"] >= "2004-01-01"]
_cov["year"] = _cov["anndats"].dt.year

tab = _cov.groupby("year").agg(事件数=("eid", "size"))
for tag, lab in [("0_1", "[d, d+1]"), ("m1_1", "[d-1, d+1]")]:
    tab[f"覆盖率 {lab}"] = (_cov.groupby("year")[f"llm_n_{tag}"].apply(lambda s: (s > 0).mean()) * 100).round(1)

n_all = len(_cov)
n_hit = int((_cov["llm_n_0_1"] > 0).sum())
print(f"2004 年及以后的 baseline 回归样本: {n_all:,} 个事件 | "
      f"有 LLM signal {n_hit:,} ({n_hit / n_all:.1%})\n")
print(tab.to_string())

2004 年及以后的 baseline 回归样本: 219,289 个事件 | 有 LLM signal 140,608 (64.1%)

        事件数  覆盖率 [d, d+1]  覆盖率 [d-1, d+1]
year                                     
2004  10495          40.5            40.9
2005  10992          42.6            43.0
2006  11164          43.8            44.1
2007  10993          47.0            47.3
2008  10618          50.7            51.1
2009  10141          53.8            54.1
2010  10319          56.3            56.6
2011   9762          59.6            60.1
2012   9535          63.6            64.2
2013   9661          67.0            67.4
2014   9458          70.9            71.3
2015   9566          73.7            74.1
2016   9472          78.4            78.8
2017   9129          84.3            84.6
2018   8527          89.6            89.9
2019   8551          66.5            66.6
2020   8898          68.7            68.8
2021   9379          70.7            70.9
2022   9691          73.1            73.4
2023   9830          75.1            75.5
2024  

---

# 5. LLM signal 单独进回归（不含 moderator）

## 5.1 为什么要单独跑

$\beta^{w}_4$ 在 §2.3 / §3.3 里是嵌在带 moderator 的方程中估的。
单独跑一版可以回答一个更基本的问题：**在控制了盈余意外与 10 个公司特征之后，
LLM signal 自己对公告窗口与漂移窗口的 CAR 有没有作用。**
这是判断 $\beta^{w}_5$ 有没有意义的前提 —— 若 LLM signal 本身对 CAR 无解释力，
谈它被 moderator 调节就没有基础。

### 设定 (1)：LLM 主效应

$$CAR^{w}_{i,d}=\alpha^{w}+\beta^{w}_1 SUE^{rank}_{i,d}+\beta^{w}_2 LLM_{i,d}
+\sum_k\gamma^{w}_k X_{k,i,d}+\sum_k\delta^{w}_k\left(X_{k,i,d}\times SUE^{rank}_{i,d}\right)
+\sum_k\phi^{w}_k\left(X_{k,i,d}\times LLM_{i,d}\right)+\eta^{w}_t+\psi^{w}_j+\varepsilon^{w}_{i,d}$$

### 设定 (2)：再加 $SUE\times LLM$

$$CAR^{w}_{i,d}=\ \cdots\ +\underbrace{\beta^{w}_3\left(SUE^{rank}_{i,d}\times LLM_{i,d}\right)}_{\text{两条信息渠道是否互补}}+\ \cdots$$

$\beta^{w}_3$ 问的是：**盈余意外的市场反应，是否随同期新闻信号的强弱而不同**。
若为正，说明新闻与盈余在同一方向上相互印证时市场反应更大。

设定、样本、固定效应、标准误与 §2.3 / §3.3 完全一致；$LLM$ 同样标准化为标准差单位。
`pred` 的训练目标是 open-to-open 收益，**O2O 列是同源口径**，C2C 并列报出作对照。

## 5.2 结果

In [19]:
res_llm_only, res_llm_only_x = {}, {}
for tag, lab in LLM_WINDOWS.items():
    d_ = df[df[f"llm_cum_{tag}"].notna()].copy()
    INTER_ = add_ctrl_x(d_, "sue_rank", "sue")
    z = d_[f"llm_{tag}"]
    LL = [f"llm_{tag}"]
    for c in CTRL:
        nm = f"{c}_x_llm"
        d_[nm] = (d_[c].astype("float64") - d_[c].astype("float64").mean()) * z
        LL.append(nm)
    d_["sue_x_llm"] = d_["sue_rank"] * z
    LABELS["sue_x_llm"] = "  SUE x LLM"

    xs1 = ["sue_rank", f"llm_{tag}"] + CTRL + INTER_ + LL[1:]
    xs2 = xs1 + ["sue_x_llm"]
    res_llm_only[tag] = ({k: run(y, xs1, data=d_) for k, y in CARS.items()}, xs1)
    res_llm_only_x[tag] = ({k: run(y, xs2, data=d_) for k, y in CARS.items()}, xs2)

    print(pub_table(res_llm_only[tag][0], ["sue_rank", f"llm_{tag}"], LABELS,
                    title=f"Table C1[{tag}]. LLM signal alone, window {lab}",
                    notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                          "LLM signal = compounded prediction over the window, standardised to mean 0 and s.d. 1.\n"
                          "pred is trained on the open-to-open return, so the O2O columns are the matched convention.\n"
                          "All ten controls enter directly, interacted with SUE, and interacted with LLM."))
    print()
    print(pub_table(res_llm_only_x[tag][0], ["sue_rank", f"llm_{tag}", "sue_x_llm"], LABELS,
                    title=f"Table C2[{tag}]. Adding SUE x LLM, window {lab}",
                    notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01"))
    print()
    del d_
    gc.collect()

print("LLM main effect across the two windows")
rows = []
for tag, lab in LLM_WINDOWS.items():
    r = res_llm_only[tag][0]
    row = {"Window": f"{lab} ({tag})", "N": f"{r[('ANN', 'O2O')].n:,}"}
    for w, c in CARS:
        b, se, t, p = grab(r[(w, c)], f"llm_{tag}")
        row[f"{w} {c}"] = f"{b:+.4f}{stars(p)} ({t:+.1f})"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

print("\nSUE x LLM across the two windows")
rows = []
for tag, lab in LLM_WINDOWS.items():
    r = res_llm_only_x[tag][0]
    row = {"Window": f"{lab} ({tag})", "N": f"{r[('ANN', 'O2O')].n:,}"}
    for w, c in CARS:
        b, se, t, p = grab(r[(w, c)], "sue_x_llm")
        row[f"{w} {c}"] = f"{b:+.4f}{stars(p)} ({t:+.1f})"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

Table C1[0_1]. LLM signal alone, window [d, d+1]
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0776***     0.0152***     0.0713***     0.0232***
                                    (0.0009)      (0.0022)      (0.0008)      (0.0023)
LLM signal (1 s.d.)                0.0185***       0.0014*     0.0183***     0.0027***
                                    (0.0004)      (0.0008)      (0.0003)      (0.0008)
--------------------------------------------------------------------------------------
Year / Month / DoW FE                    Yes           Yes           Yes           Yes
Industry FE (FF10)                       Yes           Yes           Yes           Yes
Controls                                 Yes           Yes           Yes         

Table C1[m1_1]. LLM signal alone, window [d-1, d+1]
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0811***     0.0154***     0.0745***     0.0238***
                                    (0.0009)      (0.0022)      (0.0008)      (0.0023)
LLM signal (1 s.d.)                0.0157***       0.0015*     0.0158***     0.0023***
                                    (0.0004)      (0.0008)      (0.0003)      (0.0008)
--------------------------------------------------------------------------------------
Year / Month / DoW FE                    Yes           Yes           Yes           Yes
Industry FE (FF10)                       Yes           Yes           Yes           Yes
Controls                                 Yes           Yes           Yes      

**全系数表**

In [20]:
# 全系数版
for tag, lab in LLM_WINDOWS.items():
    r, xs = res_llm_only[tag]
    print(pub_table(r, ["sue_rank", f"llm_{tag}", "__sep__"] + xs[2:], LABELS,
                    title=f"Table C1[{tag}]-full. All coefficients, window {lab}",
                    notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01"))
    print()

Table C1[0_1]-full. All coefficients, window [d, d+1]
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0776***     0.0152***     0.0713***     0.0232***
                                    (0.0009)      (0.0022)      (0.0008)      (0.0023)
LLM signal (1 s.d.)                0.0185***       0.0014*     0.0183***     0.0027***
                                    (0.0004)      (0.0008)      (0.0003)      (0.0008)
Controls and interactions:                                                            
                                                                                      
SIZE decile                        0.0008***       -0.0001     0.0007***        0.0009
                                    (0.0003)      (0.0007)      (0.0002)    

## 5.3 解读

### LLM 主效应 $\beta^{w}_2$（每 1 个标准差）

| 窗口 | ANN C2C | DRIFT C2C | ANN O2O | DRIFT O2O |
|---|---|---|---|---|
| $[d,\ d+1]$ | +0.0185\*\*\* (52.5) | +0.0014\* (1.7) | +0.0183\*\*\* (58.8) | **+0.0027\*\*\*** (3.2) |
| $[d-1,\ d+1]$ | +0.0157\*\*\* (41.5) | +0.0015\* (1.8) | +0.0158\*\*\* (48.1) | **+0.0023\*\*\*** (2.8) |

**① ANN 上的巨大系数含机械成分，不能当作发现。**
$t$ 值 40–60，LLM signal 每高 1 个标准差公告窗口 CAR 高 1.6–1.9 个百分点。
但 `pred` 的训练目标就是 open$_t\to$open$_{t+1}$ 的收益，而 $CAR^{ANN}$（O2O）覆盖的正是同一段区间 ——
预测虽是滚动样本外的，被解释变量却与训练目标重合。这一列衡量的主要是"模型预测得准"，
不是"新闻信息推动了公告反应"。

**② DRIFT 上的系数才是干净的证据。** 窗口 $[2,61]$ 与训练目标完全不重叠：
O2O 两个窗口都是 1% 显著（+0.0027\*\*\* / +0.0023\*\*\*），C2C 是 10% 边际显著。
读作：**LLM signal 每高 1 个标准差，公告后 60 个交易日还多出约 0.25 个百分点的异常收益**。

这说明新闻里被 LLM 提取到的信息**没有在公告当期被完全消化**，会延续到之后的漂移里 ——
这正是 PEAD 式欠反应在新闻信号上的体现，也为 §2.3 / §3.3 的调节检验提供了前提。

### $SUE\times LLM$：两条信息渠道互补

| 窗口 | ANN C2C | DRIFT C2C | ANN O2O | DRIFT O2O |
|---|---|---|---|---|
| $[d,\ d+1]$ | +0.0030\*\*\* (2.9) | +0.0057\*\* (2.2) | +0.0014 (1.6) | **+0.0079\*\*\*** (3.1) |
| $[d-1,\ d+1]$ | +0.0018\* (1.7) | +0.0081\*\*\* (3.2) | +0.0008 (0.9) | **+0.0098\*\*\*** (3.9) |

**DRIFT 上四个格子全部显著为正**，两个窗口、两个口径一致。
读作：**新闻信号越强，盈余意外带来的后续漂移越大**。

以 O2O、$[d,d+1]$ 为例：LLM signal 处于均值时 $\beta_1^{DRIFT}=0.0232$；
每高 1 个标准差，SUE 的漂移斜率再增加 0.0079，即高约三分之一。

经济含义是**两条渠道相互放大而非替代**：盈余意外与同期新闻指向同一方向时，
市场需要消化的信息更多，欠反应也更严重。这与 §3.3 的结论一致 ——
在那里 `SUE × ATT` 加入 LLM 后系数几乎不变，说明 SUE 与 LLM 携带的是不同的信息。

ANN 上 O2O 两个窗口都不显著，C2C 弱显著，方向为正但强度有限。

## 5.4 设定 (3)：信息量而非信号方向

`llm_cum` 是**有方向的**收益预测，设定 (1)(2) 估的是"对新闻内容的定向欠反应"：
signal 为正 → 漂移为正，signal 为负 → 漂移为负。

"信息越庞杂、市场越消化不完"是另一个假说 —— 它讲的是**信息量**，
预测漂移的**幅度**（不分方向）随信息量上升。带符号的变量在定义上分不开这两者：
$+2$ 与 $-2$ 个标准差在信息量上相同，在设定 (1) 里却贡献相反方向的漂移。

信息量用窗口内的**新闻条数** $NNEWS = $ `llm_n` 度量（取 $\log(1+\cdot)$ 压缩右尾），
两个设定分别对应"幅度"与"斜率"两种问法：

**(3a) 漂移的绝对幅度**

$$\bigl|CAR^{DRIFT}_{i,d}\bigr|=\alpha+\lambda_1 \log(1+NNEWS_{i,d})+\lambda_2 SUE^{rank}_{i,d}
+\sum_k\gamma_k X_k+\eta_t+\psi_j+\varepsilon$$

$\lambda_1>0$ 即"新闻越多，漂移的绝对幅度越大"。

**(3b) 盈余意外的漂移斜率**

$$CAR^{w}_{i,d}=\alpha+\beta_1 SUE^{rank}_{i,d}+\beta_2 \log(1+NNEWS_{i,d})
+\underbrace{\beta_3\left(SUE^{rank}_{i,d}\times \log(1+NNEWS_{i,d})\right)}_{\text{信息量是否加剧欠反应}}
+\sum_k\gamma_k X_k+\sum_k\delta_k\left(X_k\times SUE^{rank}\right)+\eta_t+\psi_j+\varepsilon$$

这是 HLT distraction 检验的直接类比 —— 把"同日别家公司的公告数"换成"本公司自己的新闻数"。
$\beta^{DRIFT}_3>0$ 即"同一份盈余意外，伴随的新闻越多，市场越消化不完"。

样本与设定 (1)(2) 相同（窗口内有新闻的事件），$\log(1+NNEWS)$ 已中心化。

In [21]:
res_load_a, res_load_b = {}, {}
for tag, lab in LLM_WINDOWS.items():
    d_ = df[df[f"llm_cum_{tag}"].notna()].copy()
    d_["nnews"] = np.log1p(d_[f"llm_n_{tag}"].astype("float64"))
    d_["nnews"] = d_["nnews"] - d_["nnews"].mean()
    d_["sue_x_nnews"] = d_["sue_rank"] * d_["nnews"]
    LABELS["nnews"] = "log(1 + NNEWS)"
    LABELS["sue_x_nnews"] = "  SUE x log(1+NNEWS)"
    INTER_ = add_ctrl_x(d_, "sue_rank", "sue")

    # (3a) 漂移的绝对幅度
    for k, col in CARS.items():
        if k[0] == "DRIFT":
            d_[f"abs_{col}"] = d_[col].abs()
    ABS = {k: f"abs_{v}" for k, v in CARS.items() if k[0] == "DRIFT"}
    r_a = {k: run(y, ["nnews", "sue_rank"] + CTRL, data=d_) for k, y in ABS.items()}
    res_load_a[tag] = r_a

    # (3b) 斜率
    xs = ["sue_rank", "nnews", "sue_x_nnews"] + CTRL + INTER_
    r_b = {k: run(y, xs, data=d_) for k, y in CARS.items()}
    res_load_b[tag] = (r_b, xs)

    print(pub_table(r_a, ["nnews", "sue_rank"], LABELS,
                    title=f"Table C3a[{tag}]. |CAR drift| on news volume, window {lab}",
                    notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                          "Dependent variable: absolute value of the drift-window CAR.\n"
                          "log(1 + NNEWS) is demeaned. A positive coefficient means more news, larger drift in absolute size."))
    print()
    print(pub_table(r_b, ["sue_rank", "nnews", "sue_x_nnews"], LABELS,
                    title=f"Table C3b[{tag}]. SUE x news volume, window {lab}",
                    notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01\n"
                          "The direct analogue of the HLT distraction test, with the firm's own news count in place\n"
                          "of the number of other firms announcing the same day."))
    print()
    del d_
    gc.collect()

print("log(1+NNEWS) on |CAR drift|")
rows = []
for tag, lab in LLM_WINDOWS.items():
    r = res_load_a[tag]
    k0 = list(r)[0]
    row = {"Window": f"{lab} ({tag})", "N": f"{r[k0].n:,}"}
    for w, c in r:
        b, se, t, p = grab(r[(w, c)], "nnews")
        row[f"|{w}| {c}"] = f"{b:+.4f}{stars(p)} ({t:+.1f})"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

print("\nSUE x log(1+NNEWS)")
rows = []
for tag, lab in LLM_WINDOWS.items():
    r = res_load_b[tag][0]
    row = {"Window": f"{lab} ({tag})", "N": f"{r[('ANN', 'O2O')].n:,}"}
    for w, c in CARS:
        b, se, t, p = grab(r[(w, c)], "sue_x_nnews")
        row[f"{w} {c}"] = f"{b:+.4f}{stars(p)} ({t:+.1f})"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

Table C3a[0_1]. |CAR drift| on news volume, window [d, d+1]
                                         (1)           (2)
                                   DRIFT C2C     DRIFT O2O
----------------------------------------------------------
log(1 + NNEWS)                     0.0089***     0.0090***
                                    (0.0015)      (0.0015)
SUE rank [0,1]                        0.0014        0.0016
                                    (0.0019)      (0.0018)
----------------------------------------------------------
Year / Month / DoW FE                    Yes           Yes
Industry FE (FF10)                       Yes           Yes
Controls                                 Yes           Yes
Observations                         140,608       138,789
Within R-squared                      0.0541        0.0611
Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01
Dependent variable: absolute value of the drift-window CAR.
log(1 + NNEWS) is 

Table C3a[m1_1]. |CAR drift| on news volume, window [d-1, d+1]
                                         (1)           (2)
                                   DRIFT C2C     DRIFT O2O
----------------------------------------------------------
log(1 + NNEWS)                     0.0091***     0.0094***
                                    (0.0015)      (0.0014)
SUE rank [0,1]                        0.0016        0.0017
                                    (0.0019)      (0.0018)
----------------------------------------------------------
Year / Month / DoW FE                    Yes           Yes
Industry FE (FF10)                       Yes           Yes
Controls                                 Yes           Yes
Observations                         141,325       139,488
Within R-squared                      0.0542        0.0611
Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01
Dependent variable: absolute value of the drift-window CAR.
log(1 + NNEWS) 

**全系数表**

In [22]:
# 全系数版
for tag, lab in LLM_WINDOWS.items():
    r, xs = res_load_b[tag]
    print(pub_table(r, ["sue_rank", "nnews", "sue_x_nnews", "__sep__"] + xs[3:], LABELS,
                    title=f"Table C3b[{tag}]-full. All coefficients, window {lab}",
                    notes="Standard errors clustered by announcement date in parentheses. * p<0.10, ** p<0.05, *** p<0.01"))
    print()

Table C3b[0_1]-full. All coefficients, window [d, d+1]
                                         (1)           (2)           (3)           (4)
                                     ANN C2C     DRIFT C2C       ANN O2O     DRIFT O2O
--------------------------------------------------------------------------------------
SUE rank [0,1]                     0.0918***     0.0154***     0.0854***     0.0244***
                                    (0.0009)      (0.0022)      (0.0008)      (0.0022)
log(1 + NNEWS)                    -0.0207***       -0.0035    -0.0202***       -0.0025
                                    (0.0016)      (0.0044)      (0.0014)      (0.0044)
  SUE x log(1+NNEWS)               0.0369***       0.0150*     0.0369***        0.0103
                                    (0.0029)      (0.0082)      (0.0025)      (0.0082)
Controls and interactions:                                                            
                                                                           

## 5.5 设定 (3) 的解读

### (3a) 新闻越多，漂移的绝对幅度越大

| 窗口 | N | \|DRIFT\| C2C | \|DRIFT\| O2O |
|---|---|---|---|
| $[d,\ d+1]$ | 140,608 | **+0.0089\*\*\*** (5.8) | **+0.0090\*\*\*** (6.1) |
| $[d-1,\ d+1]$ | 141,325 | **+0.0091\*\*\*** (6.2) | **+0.0094\*\*\*** (6.8) |

四个格子全部 1% 显著，$t$ 值 5.8–6.8。新闻条数每翻一倍（$\log$ 增加 0.69），
漂移窗口 CAR 的绝对幅度增加约 0.62 个百分点。**方向上支持"信息越多、漂移越大"。**

**但这一格有一个混淆必须说清楚：$|CAR|$ 会机械地捡到波动率。**
新闻多的公司本身波动更大，波动大则 $|CAR|$ 大，与欠反应无关。
回归里控住了 EVOL（盈余波动）与 TURN（换手率），但**没有控住收益波动率本身**。
所以这一格只能作为提示，不能单独作为证据 —— 要做实需要加入公告前的日收益波动率作控制变量，
或改用波动率标准化的漂移。

### (3b) 新闻越多，盈余意外的反应越强

| 窗口 | ANN C2C | DRIFT C2C | ANN O2O | DRIFT O2O |
|---|---|---|---|---|
| $[d,\ d+1]$ | **+0.0369\*\*\*** (12.8) | +0.0150\* (1.8) | **+0.0369\*\*\*** (15.0) | +0.0103 (1.3) |
| $[d-1,\ d+1]$ | **+0.0324\*\*\*** (12.1) | **+0.0176\*\*** (2.3) | **+0.0336\*\*\*** (14.6) | +0.0115 (1.5) |

**① ANN 上极强。** 新闻条数每翻一倍，SUE 的公告窗口斜率增加约 0.025
（$0.69\times0.037$），相对 $\beta_1\approx0.077$ 是**三分之一**。
读作：伴随新闻越多的盈余意外，市场当期反应越强。

**② DRIFT 上为正但弱。** C2C 两个窗口是 10% 与 5% 显著，O2O 都不显著（$t=1.3$–$1.5$）。
方向与"信息越多、越消化不完"一致，强度不足。

**③ 与 HLT 的 NRANK 符号相反，这不是矛盾。**
HLT 的 NRANK 数的是**同日别家公司**的公告数 —— 那是把注意力从本公司**引开**，
所以预测 ANN 更弱、DRIFT 更强。这里数的是**本公司自己**的新闻数 ——
那是本公司信息**更多**，所以 ANN 更强。两者度量的是不同的东西：
一个是注意力被瓜分，一个是信息强度。

$\S3.2$ 的 `SUE × ATT`（+0.0022\*\*\*，ANN）与这里的 `SUE × log(1+NNEWS)`（+0.037\*\*\*，ANN）
方向相同、量级差一个数量级，也说明两者不是同一回事。

### 合起来看

回到最初的问题 —— "LLM 信息说明信息更庞杂，所以加剧了漂移吗"：

- **漂移的绝对幅度确实随新闻量上升**（(3a)，$t$ 6 左右），但有波动率混淆，需要进一步控制才能坐实
- **盈余意外的漂移斜率也随新闻量上升**（(3b) DRIFT），方向对，但只有 C2C 达到常规显著性
- **最强的是公告窗口**：新闻越多，市场当期对盈余意外的反应越强（$t$ 12–15）。
  这更像"信息更充分 → 定价更充分"，而不是"信息过载 → 处理不过来"

所以信息量假说得到的是**部分支持**：新闻量与漂移正相关，但主导的效应发生在公告当期而非之后。
要把"信息过载"这条做实，下一步应当控住收益波动率，并把新闻量与 SUE 的分位交叉做非线性检验。